# CUDA Notebook Launch Lab：从环境检查到编译、运行、验证与计时

这是原创的面试准备示例，**不是 Together AI 真题，也不是已确认的 Together 面试环境**。

本文件用一个小算子把执行流程接起来：
`notebook → Python → load_inline → CUDA C++ host wrapper → GPU kernel → synchronize → assert_close`。

**依赖与范围**
- 运行计算的 notebook backend 需要可见的 NVIDIA CUDA GPU；浏览器所在的电脑不必有 GPU。
- CUDA 版 PyTorch，以及与之兼容的 CUDA toolkit / NVCC、C++ compiler、Ninja、可写的编译缓存目录。
- 仅支持同形状、同设备、连续存储的 FP32 输入；不支持 broadcasting；示例只实现前向，不注册 autograd。
- 不自动安装／升级库，不修改驱动、设备权限或 `TORCH_CUDA_ARCH_LIST`。面试应先遵循提供的 harness。

**验证说明**：生成本文件的环境为 CPU-only PyTorch，没有 NVCC / NVIDIA GPU。已经检查 notebook 格式与所有 Python cell 的语法，并执行环境检查 cell；**未完成 CUDA 编译、GPU 正确性或性能实测**。下面的 GPU tests 是供你在合适环境中实际执行的测试，不是声称已通过。

## 1. 环境自检

这些命令在 notebook 的 Python backend 中执行，而不是浏览器里的 JavaScript，也不是自动在你的 Mac 本机执行。
`hostname` 有时只是容器名字，不需要知道物理服务器的 IP 才能 launch。

In [ ]:
import os
import sys
import socket
import shutil
import subprocess
from pathlib import Path
import torch
from torch.utils.cpp_extension import CUDA_HOME, is_ninja_available

print("Backend hostname:", socket.gethostname())
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("PyTorch HIP build:", getattr(torch.version, "hip", None))
print("CUDA available:", torch.cuda.is_available())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
print("CUDA_HOME:", CUDA_HOME)
print("Ninja available:", is_ninja_available())
print("CXX setting:", os.environ.get("CXX", "<default>"))
print("c++ on PATH:", shutil.which("c++"))

nvcc_path = shutil.which("nvcc")
if nvcc_path is None and CUDA_HOME is not None:
    candidate = Path(CUDA_HOME) / "bin" / "nvcc"
    if candidate.is_file():
        nvcc_path = str(candidate)
print("NVCC:", nvcc_path)
if nvcc_path:
    p = subprocess.run([nvcc_path, "--version"], capture_output=True, text=True, timeout=15)
    print(p.stdout or p.stderr)

DEVICE = None
if torch.cuda.is_available():
    # Use the currently selected, visible device; do not assume a physical GPU ID.
    DEVICE = torch.device("cuda", torch.cuda.current_device())
    props = torch.cuda.get_device_properties(DEVICE)
    print("Selected device:", DEVICE)
    print("GPU name:", props.name)
    print("Compute capability:", torch.cuda.get_device_capability(DEVICE))
    print("Device memory (GiB):", round(props.total_memory / 2**30, 2))
else:
    print("No CUDA device is visible to this Python process. Resolve the runtime before compiling.")


## 2. 定义 CUDA 源码和 host wrapper

`add_kernel_f32` 是设备端 kernel。`add_cuda` 是 CPU 侧入口：验证输入、分配 GPU 输出、取当前 stream，并用 `<<<...>>>` 提交 kernel。

`blocks` 最多取 4096，只是这个基线的保守选择，不是最佳调参结果；grid-stride loop 负责覆盖更大输入。空 tensor 在 host 侧返回，不提交零 block 的 launch。

`C10_CUDA_KERNEL_LAUNCH_CHECK()` 检查 launch 报错；**不是完成同步，也不能代替执行后的数值验证**。

In [ ]:
CPP_DECL = "at::Tensor add_cuda(at::Tensor x, at::Tensor y);"

CUDA_SOURCE = r"""
#include <torch/types.h>
#include <cuda_runtime.h>
#include <c10/cuda/CUDAGuard.h>
#include <c10/cuda/CUDAStream.h>
#include <c10/cuda/CUDAException.h>
#include <cstdint>

__global__ void add_kernel_f32(
    const float* x, const float* y, float* out, int64_t n) {
    const int64_t start =
        static_cast<int64_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    const int64_t step =
        static_cast<int64_t>(gridDim.x) * blockDim.x;
    for (int64_t i = start; i < n; i += step) {
        out[i] = x[i] + y[i];
    }
}

at::Tensor add_cuda(at::Tensor x, at::Tensor y) {
    TORCH_CHECK(x.is_cuda() && y.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(x.device() == y.device(), "inputs must use the same device");
    TORCH_CHECK(x.scalar_type() == at::kFloat && y.scalar_type() == at::kFloat,
                "this demo supports FP32 only");
    TORCH_CHECK(x.sizes() == y.sizes(), "input shapes must match; no broadcasting");
    TORCH_CHECK(x.is_contiguous() && y.is_contiguous(), "inputs must be contiguous");
    TORCH_CHECK(!x.requires_grad() && !y.requires_grad(),
                "forward-only demo: autograd is not registered");

    const c10::cuda::CUDAGuard device_guard(x.device());
    auto out = at::empty_like(x);
    const int64_t n = x.numel();
    if (n == 0) {
        return out;
    }

    constexpr int threads = 256;
    const int64_t needed = (n - 1) / threads + 1;
    const int blocks = static_cast<int>(needed < 4096 ? needed : 4096);
    cudaStream_t stream = c10::cuda::getCurrentCUDAStream(x.get_device()).stream();

    add_kernel_f32<<<blocks, threads, 0, stream>>>(
        x.data_ptr<float>(), y.data_ptr<float>(), out.data_ptr<float>(), n);
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return out;
}
"""


## 3. 编译并加载

PyTorch 将字符串写为源文件，用编译器编译，再生成 Python 可调用的模块。通常第一次执行比后续调用慢；不要把首次编译混入稳态性能测量。

模块名包含源码 hash，方便明确当前加载的版本。没有手动硬编码 `sm_90` / `sm_100`。已有的编译架构环境设置仍可能影响构建，应核对而不是盲目覆盖。

In [ ]:
import hashlib
from torch.utils.cpp_extension import load_inline

if torch.version.cuda is None or getattr(torch.version, "hip", None) is not None:
    raise RuntimeError("Use an NVIDIA CUDA PyTorch runtime for this CUDA C++ demo.")
if DEVICE is None:
    raise RuntimeError("No visible CUDA GPU. Ask for/select the intended GPU runtime first.")
if CUDA_HOME is None or nvcc_path is None:
    raise RuntimeError("CUDA toolkit/NVCC was not located. A working torch CUDA wheel alone is insufficient.")
if not is_ninja_available():
    raise RuntimeError("Ninja is missing. In an interview, confirm the provided build path before installing anything.")

print("TORCH_CUDA_ARCH_LIST:", os.environ.get("TORCH_CUDA_ARCH_LIST", "<auto>"))
os.environ.setdefault("MAX_JOBS", "2")
source_hash = hashlib.sha256((CPP_DECL + CUDA_SOURCE).encode()).hexdigest()[:12]
ext = load_inline(
    name="nb_cuda_add_" + source_hash,
    cpp_sources=CPP_DECL,
    cuda_sources=CUDA_SOURCE,
    functions=["add_cuda"],
    extra_cflags=["-O2"],
    extra_cuda_cflags=["-O2"],
    with_cuda=True,
    verbose=True,
)
print("Extension loaded:", ext.__name__)


## 4. 真正调用、等待完成、验证数值

`ext.add_cuda` 返回输出 tensor，不等于 GPU 一定已经执行完。先 synchronize，让执行错误在这里暴露；再和 PyTorch 参考结果比较。

In [ ]:
x = torch.arange(17, dtype=torch.float32, device=DEVICE)
y = torch.ones_like(x)
out = ext.add_cuda(x, y)
torch.cuda.synchronize(DEVICE)
torch.testing.assert_close(out, x + y, rtol=1e-6, atol=1e-6)
print(out.cpu())
print("Small smoke test passed on this runtime.")


## 5. 边界与接口测试

覆盖空输入、warp/block 边界、非整除尺寸、二维和标量 tensor。错误输入要主动拒绝，不能把 FP16 指针当成 FP32 或把非连续布局当成连续布局读取。

In [ ]:
torch.manual_seed(2026)
shapes = [(n,) for n in [0, 1, 31, 32, 33, 255, 256, 257, 1001, 1_000_003]]
shapes += [(7, 13), (0, 7), ()]
for shape in shapes:
    x = torch.randn(shape, dtype=torch.float32, device=DEVICE)
    y = torch.randn_like(x)
    out = ext.add_cuda(x, y)
    torch.cuda.synchronize(DEVICE)
    torch.testing.assert_close(out, x + y, rtol=1e-6, atol=1e-6)
    assert out.shape == x.shape and out.device == x.device and out.dtype == x.dtype
print(f"Passed {len(shapes)} shape tests on this runtime.")

def expect_rejection(x, y, label):
    try:
        ext.add_cuda(x, y)
    except RuntimeError as exc:
        print(f"Expected rejection [{label}]:", str(exc).splitlines()[0])
    else:
        raise AssertionError(f"Expected rejection: {label}")

expect_rejection(torch.ones(4), torch.ones(4), "CPU tensors")
expect_rejection(torch.ones(4, device=DEVICE, dtype=torch.float16),
                 torch.ones(4, device=DEVICE, dtype=torch.float16), "FP16")
expect_rejection(torch.ones(4, device=DEVICE), torch.ones(5, device=DEVICE), "shape mismatch")
x_nc = torch.randn(3, 5, device=DEVICE).t()
y_nc = torch.randn(3, 5, device=DEVICE).t()
expect_rejection(x_nc, y_nc, "noncontiguous tensors")
expect_rejection(torch.randn(4, device=DEVICE, requires_grad=True),
                 torch.randn(4, device=DEVICE), "autograd not implemented")


## 6. 当前 stream 测试

这是补充练习：在非默认 stream 中生产输入并调用算子，检查 wrapper 是否尊重当前 stream。它不是完整的 race detector。

In [ ]:
side_stream = torch.cuda.Stream(device=DEVICE)
with torch.cuda.stream(side_stream):
    x_s = torch.randn(4097, dtype=torch.float32, device=DEVICE)
    y_s = torch.randn_like(x_s)
    out_s = ext.add_cuda(x_s, y_s)
    ref_s = x_s + y_s
side_stream.synchronize()
torch.testing.assert_close(out_s, ref_s, rtol=1e-6, atol=1e-6)
print("Non-default stream smoke test passed on this runtime.")


## 7. CUDA Events 计时：先正确，再 warm-up，再测量

下面测量重复调用这个 **operator 路径**的 CUDA-event 时间间隔。它不包含源码编译，但调用会分配输出，且很小的 kernel 可能被 host 提交节奏影响。

不能把该结果不加说明地称为“纯 kernel 指令耗时”或“HBM 实际流量”。同一输入反复调用也会产生缓存影响。逻辑有效带宽按 `2 reads + 1 write` 计算，仅是理想有用字节量。

In [ ]:
import statistics

def event_benchmark_ms(fn, warmup=20, iterations=100, repeats=5):
    if iterations <= 0 or repeats <= 0:
        raise ValueError("iterations and repeats must be positive")
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize(DEVICE)
    times = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(iterations):
            fn()
        end.record()
        end.synchronize()
        times.append(start.elapsed_time(end) / iterations)
    return statistics.median(times)

x_b = torch.randn(1 << 20, dtype=torch.float32, device=DEVICE)
y_b = torch.randn_like(x_b)
for label, fn in [("CUDA extension", lambda: ext.add_cuda(x_b, y_b)),
                  ("torch.add", lambda: torch.add(x_b, y_b))]:
    ms = event_benchmark_ms(fn)
    logical_gbps = (3 * x_b.numel() * x_b.element_size()) / (ms * 1e-3) / 1e9
    print(f"{label}: {ms * 1000:.3f} us/call; logical effective BW {logical_gbps:.2f} GB/s")


## 8. 可选：Triton 对照

只有在 notebook 中已经安装了兼容 Triton、并且练习／面试规则允许时才使用。**会写 Triton 不代表可以替代明确要求的 CUDA C++ 题**。

将 `RUN_TRITON` 改为 `True` 才执行这部分。它展示 Python 中的 launch 形式，不表示 Together 面试使用它。

In [ ]:
RUN_TRITON = False
if RUN_TRITON:
    import triton
    import triton.language as tl

    @triton.jit
    def add_triton_kernel(X, Y, Z, N, BLOCK: tl.constexpr):
        offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
        valid = offsets < N
        a = tl.load(X + offsets, mask=valid, other=0.0)
        b = tl.load(Y + offsets, mask=valid, other=0.0)
        tl.store(Z + offsets, a + b, mask=valid)

    def triton_add(x, y):
        if not (x.is_cuda and y.is_cuda and x.device == y.device):
            raise ValueError("inputs must be CUDA tensors on the same device")
        if x.dtype != torch.float32 or y.dtype != torch.float32:
            raise TypeError("this demo supports FP32 only")
        if x.shape != y.shape or not (x.is_contiguous() and y.is_contiguous()):
            raise ValueError("expected matching shapes and contiguous inputs")
        if x.requires_grad or y.requires_grad:
            raise ValueError("forward-only demo: no autograd registration")
        out = torch.empty_like(x)
        n = x.numel()
        if n:
            with torch.cuda.device(x.device):
                add_triton_kernel[(triton.cdiv(n, 256),)](x, y, out, n, BLOCK=256)
        return out

    for n in [0, 1, 33, 257, 1001, 1_000_003]:
        x_t = torch.randn(n, dtype=torch.float32, device=DEVICE)
        y_t = torch.randn_like(x_t)
        out_t = triton_add(x_t, y_t)
        torch.cuda.synchronize(DEVICE)
        torch.testing.assert_close(out_t, x_t + y_t, rtol=1e-6, atol=1e-6)
    print("Optional Triton checks passed on this runtime.")
else:
    print("Optional Triton section skipped. Set RUN_TRITON=True only in a compatible, allowed environment.")


## 9. 把“读懂答案”变成动手练习

建议先实际运行参考实现一次，然后保留环境检查、编译和测试 cells，把 kernel 内容删掉，自己重新写：
1. 向量加法 → 改成 `out[i] = max(x[i] + y[i], 0)`，同步修改参考实现。
2. 改成矩阵加行 bias + ReLU。明确定义 M、N、bias 的尺寸与广播规则。
3. 写 transpose、row sum、stable softmax。重新定义接口，增加非整除尺寸测试。
4. GEMM：先朴素正确，再 shared-memory tiling。关注 dtype、K=0、非方阵与同步。
5. 每次都按“执行与检查 → 计时 → 提出优化 → 再验证”的顺序。

不要为了练习刻意在面试共享实例上制造越界或死锁。非法访问和 device-side assert 后，可能需要重启 notebook 的计算 kernel，并重新执行初始化与构建 cells。

## 10. 分层排错

| 现象 | 先判断什么 |
|---|---|
| `torch.cuda.is_available()` 为 False | 当前 Python runtime / GPU 分配 / 设备可见性；不要先改数学索引 |
| NVCC / CUDA_HOME / Ninja 缺失 | 构建环境或使用了错误的编译路径 |
| compiler syntax error | 阅读第一个实际编译错误，核对 C++ / CUDA 签名 |
| `no kernel image is available` | 编译目标与当前 GPU 的兼容性 |
| `invalid configuration argument` | grid、block、shared-memory 资源配置 |
| `illegal memory access` / 结果错误 | 边界、索引、指针 dtype、stride、同步等 |
| 初次运行慢，之后快 | 区分编译、加载、初始化、warm-up 与稳态执行 |
| Kernel 卡住 | barrier 分支是否一致、循环是否可终止；与面试官确认重启 runtime |

## 11. 面试前可以问的英文问题

> Could you confirm whether the CUDA round uses a provided GPU-backed notebook? What GPU and CUDA/PyTorch environment will be available, and will a compilation/launch scaffold and correctness tests be provided? Is the expected implementation CUDA C++, or is Triton also allowed? Are documentation lookup and prepared boilerplate permitted?

## References

- Jupyter architecture: https://docs.jupyter.org/en/latest/projects/architecture/content-architecture.html
- PyTorch load_inline: https://docs.pytorch.org/docs/stable/cpp_extension.html
- PyTorch CUDA semantics: https://docs.pytorch.org/docs/stable/notes/cuda.html
- PyTorch custom C++/CUDA operators: https://docs.pytorch.org/tutorials/advanced/cpp_custom_ops.html
- CUDA C++ introduction: https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html
- Triton vector addition tutorial: https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html

这些官方文档解释技术机制；不证明 Together 使用此 notebook 或此题。